In [2]:
import json
import pickle
import numpy as np
import pandas as pd
import os
print(os.getcwd())

c:\Users\atom0\OneDrive\Documents\College_folder\4th\DACN_Do-Thanh-Thai\CL-TTE\playground


In [3]:
data_path = "../../data/mydata/"
train_path = os.path.join(data_path, "train.npy")
data = np.load(train_path, allow_pickle=True)
print(f"Loaded data from {train_path}, shape: {data.shape}")

Loaded data from ../../data/mydata/train.npy, shape: (900877, 6)


In [4]:
with open(os.path.join(data_path,"network_porto/porto_edges_poi_new_simplify.pkl"), 'rb') as f:
    edgeinfo = pickle.load(f)
    
sample_edge = iter(edgeinfo.values()).__next__()
print("Full edge list  :", sample_edge)
print("Total length    :", len(sample_edge))

Full edge list  : ['motorway_link', 32.3884588871153, '25503936', '4722746638', 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Total length    : 16


In [8]:
print(data[0])

[1386940210620000409
 list([10497, 3860, 2554, 2556, 6897, 1945, 1910, 2547, 3889, 9651, 9654, 4074, 9658, 8588, 3941, 9661, 10295, 9671, 3939, 3872, 3935, 6604, 3833, 2520, 3823, 3832, 6592])
 4 347 790 405]


In [11]:
trip_tt = np.array(data[:, -1])
poi_vector_length = len(edgeinfo[0][4:])
trip_poi_presence = np.zeros((len(data),poi_vector_length), dtype=bool) # (N, G) - 1 if the trip has at least 1 POI in group g, 0 otherwise

for tripidx, d in enumerate(data):
    for e in d[1]:
        poi_vector = edgeinfo[e][4:] # 
        for i in range(len(poi_vector)):
            if poi_vector[i] > 0:
                trip_poi_presence[tripidx, i] = 1

print("Trip POI presence shape:", trip_poi_presence.shape)
print("Trip travel time shape:", trip_tt.shape)
    

Trip POI presence shape: (900877, 12)
Trip travel time shape: (900877,)


In [12]:
for g in range(poi_vector_length):
    print(f"Group {g}: {np.sum(trip_poi_presence[:, g])} trips have at least 1 POI in this group")

Group 0: 507894 trips have at least 1 POI in this group
Group 1: 895780 trips have at least 1 POI in this group
Group 2: 883326 trips have at least 1 POI in this group
Group 3: 766033 trips have at least 1 POI in this group
Group 4: 856192 trips have at least 1 POI in this group
Group 5: 794315 trips have at least 1 POI in this group
Group 6: 805205 trips have at least 1 POI in this group
Group 7: 890407 trips have at least 1 POI in this group
Group 8: 811601 trips have at least 1 POI in this group
Group 9: 879312 trips have at least 1 POI in this group
Group 10: 826803 trips have at least 1 POI in this group
Group 11: 900753 trips have at least 1 POI in this group


In [15]:
from scipy import stats
from sklearn.linear_model import LinearRegression
import numpy as np

# control for trip length — residualize travel time
trip_lengths = np.array([len(d[1]) for d in data], dtype=np.float64)
trip_tt_float = trip_tt.astype(np.float64)

# safer correlation that avoids the numpy internal bug
corr = np.corrcoef(trip_lengths, trip_tt_float)[0, 1]
print(f"Length-TT correlation: {corr:.3f}")
reg = LinearRegression()
reg.fit(trip_lengths.reshape(-1, 1), trip_tt_float)
tt_residual = trip_tt_float - reg.predict(trip_lengths.reshape(-1, 1))

print(f"Residual mean: {tt_residual.mean():.2f}, std: {tt_residual.std():.2f}\n")

category_names = [
    'Group 0', 'Group 1', 'Group 2', 'Group 3',
    'Group 4', 'Group 5', 'Group 6', 'Group 7',
    'Group 8', 'Group 9', 'Group 10', 'Group 11'
]  # replace with your actual category names
print(f"{'Category':<30} {'N_with':>8} {'N_without':>10} "
      f"{'mean_with':>10} {'mean_without':>13} {'effect':>8} {'sig':>5}")
print("-" * 90)

results = {}
for g, name in enumerate(category_names):
    present   = trip_poi_presence[:, g]
    n_with    = int(present.sum())
    n_without = int((~present).sum())

    if n_with < 30 or n_without < 30:
        print(f"{name:<30} SKIPPED — insufficient contrast")
        continue

    tt_with    = tt_residual[present]
    tt_without = tt_residual[~present]

    stat, pval = stats.mannwhitneyu(
        tt_with, tt_without, alternative='two-sided'
    )

    # cast to float64 before multiplication to prevent overflow
    effect = 1.0 - (2.0 * float(stat)) / (float(n_with) * float(n_without))

    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'ns'

    results[name] = {
        'n_with':       n_with,
        'n_without':    n_without,
        'mean_with':    tt_with.mean(),
        'mean_without': tt_without.mean(),
        'effect':       effect,
        'pval':         pval,
        'sig':          sig,
    }

    print(f"{name:<30} {n_with:>8} {n_without:>10} "
          f"{tt_with.mean():>10.2f} {tt_without.mean():>13.2f} "
          f"{effect:>+8.4f} {sig:>5}")

print("\n--- Ranked by absolute effect size ---")
for name, r in sorted(results.items(),
                      key=lambda x: abs(x[1]['effect']), reverse=True):
    direction = "slower" if r['effect'] > 0 else "faster"
    print(f"{name:<30} effect={r['effect']:+.4f}  {direction}  {r['sig']}")

Length-TT correlation: 0.760
Residual mean: -0.00, std: 216.68

Category                         N_with  N_without  mean_with  mean_without   effect   sig
------------------------------------------------------------------------------------------
Group 0                          507894     392983      -0.11          0.14  +0.0415   ***
Group 1                          895780       5097       0.35        -61.14  -0.2208   ***
Group 2                          883326      17551       1.29        -64.92  -0.2224   ***
Group 3                          766033     134844       6.91        -39.25  -0.1355   ***
Group 4                          856192      44685       2.14        -41.07  -0.1228   ***
Group 5                          794315     106562       3.59        -26.75  -0.0944   ***
Group 6                          805205      95672       1.82        -15.30  -0.0598   ***
Group 7                          890407      10470       0.65        -55.62  -0.1871   ***
Group 8                   